In [ ]:
import os
import time
import pickle
from typing import Dict, List
import pandas as pd
from PIL import Image
from IPython.display import display
from itertools import product

from mosaic_pipeline import *
from evaluator import *

In [ ]:

# Define a set of feature weight combinations to evaluate
feature_combinations = [
    {"w_color": 0.6, "w_hog": 0.3, "w_edge": 0.1},      # HOG + Color + Edge
    {"w_cnn": 1.0},                                    # Pure CNN
    {"w_color": 1.0},                                  # Pure Color Histogram
    {"w_hog": 0.5, "w_color": 0.5},                    # HOG + Color
    {"w_edge": 0.3, "w_color": 0.7},                   # Edge + Color 
    {"w_cnn": 0.5, "w_color": 0.5},                    # CNN + Color
    {"w_hog": 0.1, "w_color": 0.9}                     # Low HOG + High Color 
]

# Prepare result storage
results = []

# Configuration
image_path = "000145.jpg"                  # Replace with actual main image path
gallery_image_folder = "resized_tiles"  # Replace with actual folder
tile_size = 32
output_dir = "exp_outputs"
os.makedirs(output_dir, exist_ok=True)

# Loop over all combinations and evaluate
for i, weights in enumerate(feature_combinations):
    run_name = f"experiment_{i:02d}"
    start_time = time.time()

    try:
        score, norm_weights = evaluate_weights(
            weights=weights,
            image_path=image_path,
            gallery_image_folder=gallery_image_folder,
            tile_size=tile_size,
            output_dir=output_dir,
            run_name=run_name
        )

        duration = time.time() - start_time
        results.append({
            "run": run_name,
            "weights": norm_weights,
            "SSIM": score,
            "time": round(duration, 2)
        })

    except Exception as e:
        results.append({
            "run": run_name,
            "weights": weights,
            "SSIM": None,
            "time": None,
            "error": str(e)
        })

# Display results
df = pd.DataFrame(results)
df.sort_values(by="SSIM", ascending=False, inplace=True)
print(df)

